# PRIMA NER — Run 3

## Objectif de ce run

Ce run met en œuvre les recommandations de l'encadrant après l'analyse du Run 2 :

| # | Modèle | Stratégie | Données |
|---|--------|-----------|---------|
| 1 | XLM-RoBERTa + couche de classification | **2 dernières couches dégelées** (après échec de l'encodeur entièrement gelé) | **845 entrées** (train+val fusionnés, cross-validation 5-fold) + **test_all isolé** (93 entrées, évaluation finale uniquement) |
| 2 | BiLSTM-CRF from scratch | Entraîné depuis zéro, sans pré-entraînement | **938 entrées** (train+val+test, cross-validation 5-fold complète) |

### Pourquoi ces choix ?

**XLM-RoBERTa, 2 dernières couches dégelées** :
- Le Run 2 a montré que fine-tuner *tous* les paramètres d'XLM-RoBERTa sur seulement 70 exemples (train split) conduit à une convergence lente (besoin de 20 epochs) et un risque de sur-apprentissage.
- Recommandation initiale de l'encadrant : geler complètement l'encodeur et n'entraîner qu'une **couche de classification des entités nommées**. Cette première tentative a échoué (val F1=0,10, test F1=0,07) : 8 459 paramètres entraînables sont insuffisants pour adapter les représentations génériques à la tâche NER.
- On dégèle donc les **2 derniers encoder layers**, qui encodent des informations plus spécifiques à la tâche, tout en gardant gelées les couches basses (représentations linguistiques générales).
- Compromis sur petit corpus : ~14M paramètres entraînables (~5%) → assez de capacité pour apprendre la tâche, sans le coût et l'instabilité d'un fine-tuning complet (~270M paramètres).

**BiLSTM-CRF from scratch** :
- L'encadrant recommande d'entraîner BiLSTM-CRF depuis zéro (random initialization) sur l'intégralité des 938 entrées.
- Cela constitue une baseline solide sans pré-entraînement, comparable aux systèmes classiques de NER.
- Résultat final (test F1=0,63) : conclusion fixée, ce modèle n'est plus modifié.

**Utilisation des données** :
- Dans le Run 2, le split 70/20/10 limitait le jeu d'entraînement à 70 exemples sur 100.
- Les données ont été étendues de 100 à 938 entrées (100 gold + 838 remaining).
- XLM-RoBERTa : train+val (845 entrées) pour la CV, test_all (93 entrées) strictement isolé pour l'évaluation finale — nécessaire car le fine-tuning d'un modèle pré-entraîné exige un jeu de test invisible pendant l'entraînement.
- BiLSTM-CRF : les 938 entrées participent toutes à la CV 5-fold — justifié car le modèle est entraîné depuis zéro et le vocabulaire est reconstruit à chaque fold à partir des seules données d'entraînement du fold, sans fuite d'information.

### Hyperparamètres choisis

- **XLM-RoBERTa (2 dernières couches dégelées)** : `epochs=15`, `lr=2e-5`, `batch=16`, `max_len=128`. LR réduit par rapport à la tentative frozen (5e-4) car on entraîne maintenant de vrais poids de Transformer.
- **BiLSTM-CRF** : `epochs=30`, `lr=0.001`, `batch=16`, `embed=100`, `hidden=256` — résultats finaux, non modifiés.

⚠️ Exécuter **Cell 0** en premier : installation des dépendances + **upload des 3 fichiers JSON** (`train_all.json`, `val_all.json`, `test_all.json`).

---

## 中文说明

### 本次 Run 的目标

本次实验落实老师在 Run 2 分析后给出的建议：

| # | 模型 | 策略 | 数据 |
|---|------|------|------|
| 1 | XLM-RoBERTa + 分类层 | **解冻最后 2 层**（编码器完全冻结的尝试已失败） | **845 条**（train+val 合并，5 折交叉验证）+ **test_all 独立**（93 条，仅用于最终评估） |
| 2 | BiLSTM-CRF from scratch | 从零开始训练，无预训练 | **938 条**（train+val+test 全部参与 5 折交叉验证） |

### 为什么这样选择？

**XLM-RoBERTa，解冻最后 2 层**：
- Run 2 显示，在仅 70 条训练样本上完全微调 XLM-RoBERTa 的所有参数，收敛慢（需要 20 个 epoch）且容易过拟合。
- 老师最初的建议是：完全冻结编码器，只训练一个命名实体分类层。第一次尝试失败了（val F1=0.10，test F1=0.07）：8,459 个可训练参数远远不够，单层线性映射无法把通用语言表示适配到 NER 任务。
- 因此改为解冻**最后 2 个 encoder layer**——这些层编码的是更偏任务相关的信息，而底层（编码通用语言特征）依然保持冻结。
- 这是小数据集上的折中方案：约 1400 万可训练参数（约 5%），足够学会任务本身，又不必承担完整微调（约 2.7 亿参数）的成本和不稳定性。

**BiLSTM-CRF from scratch**：
- 老师建议 BiLSTM-CRF 从零开始（随机初始化）在全部 938 条数据上训练。
- 这构成了一个不依赖预训练的稳健 baseline，可与传统 NER 系统对比。
- 最终结果（test F1=0.63）已经确定，该模型不再调整。

**数据使用策略**：
- Run 2 中 70/20/10 的划分把训练集限制在 100 条里的 70 条。
- 数据已从 100 条扩展到 938 条（100 条 gold + 838 条 remaining）。
- XLM-RoBERTa：train+val（845 条）参与 CV，test_all（93 条）完全隔离仅作最终评估——预训练模型 fine-tune 时必须保证测试集在训练过程中不可见。
- BiLSTM-CRF：938 条全部参与 5 折 CV——合理，因为模型从零训练，词表在每个 fold 中仅从该 fold 的训练集构建，不存在数据泄露。

### 选定的超参数

- **XLM-RoBERTa（解冻最后 2 层）**：`epochs=15`，`lr=2e-5`，`batch=16`，`max_len=128`。相比完全冻结的尝试（5e-4），学习率调低，因为现在训练的是真实的 Transformer 权重。
- **BiLSTM-CRF**：`epochs=30`，`lr=0.001`，`batch=16`，`embed=100`，`hidden=256` —— 已是最终结果，不再修改。

⚠️ 先运行 **Cell 0**：安装依赖 + 上传 3 个 JSON 文件（`train_all.json`、`val_all.json`、`test_all.json`）。

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 0 — Installation des dépendances + upload des fichiers
# À exécuter en PREMIER dans chaque nouvelle session Colab.
#
# Fichiers à uploader (3 fichiers JSON) :
#   train_all.json, val_all.json, test_all.json
# ═══════════════════════════════════════════════════════════════

import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))

!pip install -q transformers seqeval scikit-learn
print('Dépendances installées.')

# ── Upload des fichiers de données ──────────────────────────────
# Uploader train_all.json, val_all.json, test_all.json
from google.colab import files
print('\nVeuillez uploader les 3 fichiers : train_all.json, val_all.json, test_all.json')
uploaded = files.upload()

# Vérification des fichiers uploadés
import os
for fname in ['train_all.json', 'val_all.json', 'test_all.json']:
    if fname in uploaded:
        size = len(uploaded[fname])
        print(f'  ✓ {fname} ({size/1024:.0f} KB)')
    else:
        print(f'  ✗ {fname} MANQUANT — relancer cette cellule')

# Écrire les fichiers uploadés dans /content/ (répertoire de travail Colab)
for fname, data in uploaded.items():
    with open(f'/content/{fname}', 'wb') as f:
        f.write(data)
print('\nFichiers disponibles dans /content/')


GPU disponible : True
GPU : Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Dépendances installées.

Veuillez uploader les 3 fichiers : train_all.json, val_all.json, test_all.json


Saving test_all.json to test_all.json
Saving train_all.json to train_all.json
Saving val_all.json to val_all.json
  ✓ train_all.json (1997 KB)
  ✓ val_all.json (561 KB)
  ✓ test_all.json (285 KB)

Fichiers disponibles dans /content/


## Cell 1a — Configuration & Imports

Imports, hyperparamètres, label scheme, répertoires.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — XLM-RoBERTa (2 DERNIÈRES COUCHES DÉGELÉES + couche de classification)
# Cross-validation 5-fold sur 845 entrées (train + val)
#
# Stratégie :
#   - Tentative précédente (encodeur entièrement gelé) : échec
#     (val F1=0.10, test F1=0.07) — la tête linéaire seule (8 459 params)
#     ne peut pas adapter les représentations génériques à la tâche NER.
#   - On dégèle donc les 2 derniers encoder layers d'XLM-RoBERTa
#     (requires_grad=True), le reste de l'encodeur reste gelé.
#   - On garde la couche linéaire de classification NER (768 → 11 labels),
#     entraînée conjointement avec les 2 couches dégelées.
#   - Compromis entre fine-tuning complet (Run 2, coûteux et instable sur
#     petit corpus) et frozen total (Run 3 tentative 1, sous-paramétré).
#
# Hyperparamètres :
#   EPOCHS=15   : plus de paramètres entraînables qu'en frozen total →
#                  besoin de plus d'epochs pour converger
#   LR=2e-5     : réduit par rapport au frozen total (5e-4) car on entraîne
#                  maintenant de vrais poids de Transformer (risque de
#                  catastrophic forgetting si LR trop élevé)
#   BATCH=16    : augmenté par rapport au frozen total (8)
#   MAX_LEN=128 : validé par le Run 2 (nos entrées ≤ 100 tokens)
#   K_FOLDS=5   : 845 entrées (train+val) → cross-validation 5-fold
#                  test_all.json (93 entrées) réservé à l'évaluation finale
#   Paramètres entraînables estimés : ~14M (~5% du total)
#
# Données : train_all.json (657) + val_all.json (188) = 845 entrées pour la CV
#           test_all.json (93 entrées) : isolé, évaluation finale uniquement
# ═══════════════════════════════════════════════════════════════

# -*- coding: utf-8 -*-
"""
prima_run3_xlmroberta_partial_unfreeze.py
==========================================
XLM-RoBERTa avec les 2 derniers encoder layers dégelés + couche de
classification NER. Cross-validation 5-fold sur les 938 entrées annotées.

■ Architecture
  XLM-RoBERTa-base :
    - Embeddings + couches 0 à 9 : GELÉES (requires_grad=False)
    - Couches 10 et 11 (les 2 dernières) : DÉGELÉES (requires_grad=True)
  + Linear(768 → NUM_LABELS=11)  ← entraînée conjointement

■ Différence clé avec les tentatives précédentes
  Run 2            : fine-tuning complet (tous les paramètres XLM-R mis à jour)
  Run 3 tentative 1 : encodeur entièrement gelé (échec, sous-paramétré)
  Run 3 tentative 2 : 2 dernières couches dégelées (ce script)

■ Protocole d'évaluation
  Cross-validation 5-fold sur 845 entrées (train_all + val_all).
  test_all.json (93 entrées) strictement isolé — évaluation finale uniquement.
  Métriques CV : F1 moyen ± écart-type sur les 5 folds.
  Test final : meilleur fold (F1 validation le plus élevé) évalué sur test_all.

■ Outputs
  logs/fold_{k}/training_log.jsonl   → loss + F1 par epoch
  logs/fold_{k}/results.json         → métriques du fold
  logs/cv_summary.json               → résumé cross-validation
  model/fold_{k}/                    → meilleur modèle du fold
"""

import json
import os
import time
from datetime import datetime
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)


# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_NAME = "xlm-roberta-base"

# Données : train_all (657) + val_all (188) = 845 pour la CV 5-fold
#            test_all (93) isolé → évaluation finale
DATA_DIR   = "/content"  # fichiers uploadés via Cell 0

LOG_DIR    = "/content/run3_xlmroberta_partial_unfreeze/logs"
MODEL_DIR  = "/content/run3_xlmroberta_partial_unfreeze/model"

# ── Hyperparamètres (justification dans l'en-tête de cellule) ─────────────────
EPOCHS       = 15      # Plus de paramètres entraînables que le frozen total → plus d'epochs
BATCH_SIZE   = 16      # Augmenté par rapport à la tentative frozen (8)
LR           = 2e-5    # Réduit par rapport au frozen total (5e-4) : on entraîne de vrais poids Transformer
MAX_LEN      = 128     # Validé par le Run 2 : nos entrées tiennent dans 128 tokens
K_FOLDS      = 5       # Cross-validation 5-fold sur 845 entrées (train+val uniquement)
N_UNFROZEN   = 2       # Nombre de derniers encoder layers à dégeler
SEED         = 42

# BIO label scheme
LABELS = ["auteur", "ouvrage", "material", "date", "etat"]
LABEL2ID = {"O": 0}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID)
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID)
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)  # = 11

os.makedirs(LOG_DIR,   exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device : {device}")
print(f"[Config] Modèle : {MODEL_NAME} (2 dernières couches DÉGELÉES, N_UNFROZEN={N_UNFROZEN})")
print(f"[Config] Epochs={EPOCHS}, LR={LR}, Batch={BATCH_SIZE}, MaxLen={MAX_LEN}, K={K_FOLDS}")



[Config] Device : cuda
[Config] Modèle : xlm-roberta-base (2 dernières couches DÉGELÉES, N_UNFROZEN=2)
[Config] Epochs=15, LR=2e-05, Batch=16, MaxLen=128, K=5


## Cell 1b — Classes & Fonctions

`json_to_bio`, `NERDataset`, `PartialUnfreezeXLMRoBERTaNER`, `evaluate`.

In [ ]:
# ── 1. Conversion JSON → BIO (identique au Run 2) ────────────────────────────
def json_to_bio(entries):
    """
    Convertit les entrées JSON en séquences BIO au niveau des mots.
    Identique au Run 2 : segmentation par espaces + alignement sur le premier
    caractère de chaque mot. Pas de problème de reconstruction de subwords
    (géré dans NERDataset via word_ids).
    """
    samples = []
    for entry in entries:
        raw   = entry["raw"]
        spans = entry.get("spans", [])
        char_labels = ["O"] * len(raw)
        for span in spans:
            text  = span["text"]
            label = span["label"]
            idx   = raw.find(text)
            if idx == -1:
                continue
            char_labels[idx] = f"B-{label}"
            for i in range(idx + 1, idx + len(text)):
                char_labels[i] = f"I-{label}"
        tokens, labels = [], []
        pos = 0
        for word in raw.split():
            idx = raw.find(word, pos)
            if idx == -1:
                tokens.append(word); labels.append("O")
                pos += len(word); continue
            tokens.append(word)
            labels.append(char_labels[idx])
            pos = idx + len(word)
        samples.append((tokens, labels))
    return samples


# ── 2. Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    """
    Même logique que Run 2 : alignement subword → word via word_ids().
    Compatible avec l'encodeur gelé (les inputs sont les mêmes).
    """
    def __init__(self, samples, tokenizer, max_len):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens, labels = self.samples[idx]
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        word_ids  = encoding.word_ids()
        label_ids = []
        prev_word = None
        for wid in word_ids:
            if wid is None:
                label_ids.append(-100)
            elif wid != prev_word:
                label_ids.append(LABEL2ID[labels[wid]])
            else:
                l = labels[wid]
                if l.startswith("B-"):
                    l = "I-" + l[2:]
                label_ids.append(LABEL2ID.get(l, 0))
            prev_word = wid
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(label_ids, dtype=torch.long),
        }


# ── 3. Modèle : XLM-RoBERTa (2 dernières couches dégelées) + tête de classification ──
class PartialUnfreezeXLMRoBERTaNER(nn.Module):
    """
    Architecture :
      1. XLM-RoBERTa-base :
         - Embeddings + couches 0..(L-N_UNFROZEN-1) : GELÉES (requires_grad=False)
         - Les N_UNFROZEN derniers encoder layers : DÉGELÉS (requires_grad=True)
      2. Couche linéaire de classification NER (768 → 11 labels) :
         entraînée conjointement avec les couches dégelées.

    Pourquoi dégeler les 2 dernières couches plutôt que tout geler ?
      - La tentative encodeur entièrement gelé a échoué (val F1=0.10) :
        8 459 paramètres (la seule tête linéaire) sont insuffisants pour
        adapter les représentations génériques d'XLM-R à notre tâche NER.
      - Les couches hautes d'un Transformer encodent des informations plus
        spécifiques à la tâche, contrairement aux couches basses qui captent
        des informations plus générales (syntaxe, morphologie). Dégeler les
        couches hautes permet donc une adaptation à la tâche tout en
        conservant les représentations linguistiques générales apprises
        lors du pré-entraînement (couches basses gelées).
      - Compromis : ~14M paramètres entraînables (~5% du total), suffisant
        pour apprendre la tâche sans le coût et l'instabilité d'un
        fine-tuning complet (~270M paramètres, Run 2).
    """
    def __init__(self, model_name, num_labels, n_unfrozen=2, dropout=0.1):
        super().__init__()
        # Chargement de l'encodeur XLM-RoBERTa
        self.encoder = AutoModel.from_pretrained(model_name)

        # GEL de tout l'encodeur par défaut
        for param in self.encoder.parameters():
            param.requires_grad = False

        # DÉGEL des n_unfrozen derniers encoder layers
        encoder_layers = self.encoder.encoder.layer  # ModuleList des 12 layers (base)
        n_layers = len(encoder_layers)
        assert n_unfrozen <= n_layers, "n_unfrozen ne peut pas dépasser le nombre de couches"
        for layer in encoder_layers[n_layers - n_unfrozen:]:
            for param in layer.parameters():
                param.requires_grad = True

        # Couche de classification NER (toujours entraînable)
        hidden_size = self.encoder.config.hidden_size  # 768 pour xlm-roberta-base
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        # Les couches gelées ne calculent pas de gradient (requires_grad=False),
        # mais le forward pass complet doit rester dans le graphe d'autograd
        # pour que le gradient remonte jusqu'aux couches dégelées.
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Représentations contextuelles de chaque token : (batch, seq_len, 768)
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        # Projection vers l'espace des étiquettes : (batch, seq_len, 11)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # Cross-entropy, en ignorant les positions -100 (tokens spéciaux)
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
        return loss, logits


# ── 4. Fonctions d'évaluation ─────────────────────────────────────────────────
def bio_to_spans(tokens, bio):
    spans = []
    cur_text, cur_label = [], None
    for tok, tag in zip(tokens, bio):
        if tag.startswith("B-"):
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text  = [tok]; cur_label = tag[2:]
        elif tag.startswith("I-") and cur_text:
            cur_text.append(tok)
        else:
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text, cur_label = [], None
    if cur_text:
        spans.append({"text": " ".join(cur_text), "label": cur_label})
    return spans


def evaluate(model, loader, split_name="val", entries=None):
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            lab       = batch["labels"].to(device)
            _, logits = model(input_ids=input_ids, attention_mask=attn_mask)
            preds     = torch.argmax(logits, dim=-1)
            for pred_seq, true_seq in zip(preds.cpu().tolist(), lab.cpu().tolist()):
                p_tags, t_tags = [], []
                for p, t in zip(pred_seq, true_seq):
                    if t == -100: continue
                    p_tags.append(ID2LABEL[p])
                    t_tags.append(ID2LABEL[t])
                all_preds.append(p_tags)
                all_trues.append(t_tags)
    report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
    f1  = f1_score(all_trues, all_preds, zero_division=0)
    pre = precision_score(all_trues, all_preds, zero_division=0)
    rec = recall_score(all_trues, all_preds, zero_division=0)
    print(f"  [{split_name}] F1={f1:.4f}  P={pre:.4f}  R={rec:.4f}")
    print(classification_report(all_trues, all_preds, zero_division=0))
    details = []
    if entries is not None:
        for entry, gold_bio, pred_bio in zip(entries, all_trues, all_preds):
            tokens = entry["raw"].split()
            details.append({
                "entry_id":   entry["entry_id"],
                "raw":        entry["raw"],
                "gold_spans": bio_to_spans(tokens, gold_bio),
                "pred_spans": bio_to_spans(tokens, pred_bio),
            })
    return {"f1": f1, "precision": pre, "recall": rec, "report": report, "details": details}


## Cell 1c — Chargement des données & Tokenizer

Chargement train+val (845 entrées pour la CV) et test isolé (93 entrées).

In [ ]:
# ── 5. Chargement des données ────────────────────────────────────────────────
# CORRECTION : test_all.json (93 entrées) strictement isolé de la CV.
# Seuls train_all (657) + val_all (188) = 845 entrées participent aux 5 folds.
print("\n[1/5] Chargement des données...")

# -- 5a. Données CV : train + val (845 entrées) --
cv_entries = []
for split in ["train", "val"]:
    path = os.path.join(DATA_DIR, f"{split}_all.json")
    with open(path, encoding="utf-8") as f:
        part = json.load(f)
    cv_entries.extend(part)
    print(f"  {split}: {len(part)} entrées")
print(f"  Total CV (train+val) : {len(cv_entries)} entrées")

# -- 5b. Données test : isolées (93 entrées) --
test_path = os.path.join(DATA_DIR, "test_all.json")
with open(test_path, encoding="utf-8") as f:
    test_entries = json.load(f)
print(f"  test (isolé)         : {len(test_entries)} entrées")

all_samples  = json_to_bio(cv_entries)
test_samples = json_to_bio(test_entries)
all_indices  = list(range(len(cv_entries)))


# ── 6. Tokenizer ─────────────────────────────────────────────────────────────
print("\n[2/5] Chargement du tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)



[1/5] Chargement des données...
  train: 657 entrées
  val: 188 entrées
  Total CV (train+val) : 845 entrées
  test (isolé)         : 93 entrées

[2/5] Chargement du tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

## Cell 1d — Cross-validation 5-fold

Boucle d'entraînement : 5 folds × 15 epochs. Sauvegarde du meilleur modèle par fold.

In [ ]:

# ── 7. Cross-validation 5-fold ────────────────────────────────────────────────
print(f"\n[3/5] Cross-validation {K_FOLDS}-fold sur {len(cv_entries)} entrées (test isolé)...")
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

fold_results = []
cv_start = time.time()

for fold, (train_idx, val_idx) in enumerate(kf.split(all_indices), start=1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{K_FOLDS}  (train={len(train_idx)}, val={len(val_idx)})")
    print('='*60)

    fold_log_dir   = os.path.join(LOG_DIR,   f"fold_{fold}")
    fold_model_dir = os.path.join(MODEL_DIR, f"fold_{fold}")
    os.makedirs(fold_log_dir,   exist_ok=True)
    os.makedirs(fold_model_dir, exist_ok=True)

    train_samp = [all_samples[i] for i in train_idx]
    val_samp   = [all_samples[i] for i in val_idx]
    val_ents   = [cv_entries[i]  for i in val_idx]

    train_dataset = NERDataset(train_samp, tokenizer, MAX_LEN)
    val_dataset   = NERDataset(val_samp,   tokenizer, MAX_LEN)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

    # Initialisation du modèle (nouvel encodeur à chaque fold)
    model = PartialUnfreezeXLMRoBERTaNER(MODEL_NAME, NUM_LABELS, n_unfrozen=N_UNFROZEN).to(device)

    # Vérification : seuls les paramètres de la tête linéaire sont entraînables
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Paramètres entraînables : {trainable:,} / {total:,} "
          f"({100*trainable/total:.2f}%)")

    # Optimiseur : AdamW sur les seuls paramètres entraînables
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    # Entraînement
    log_path = os.path.join(fold_log_dir, "training_log.jsonl")
    best_f1  = 0.0

    with open(log_path, "w", encoding="utf-8") as logf:
        for epoch in range(1, EPOCHS + 1):
            epoch_start = time.time()
            model.train()
            total_loss = 0.0

            for batch in train_loader:
                input_ids = batch["input_ids"].to(device)
                attn_mask = batch["attention_mask"].to(device)
                labels    = batch["labels"].to(device)

                loss, _ = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                total_loss += loss.item()

                loss.backward()
                # Gradient clipping sur la tête de classification seulement
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()), 1.0
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            avg_loss   = total_loss / len(train_loader)
            epoch_time = time.time() - epoch_start
            timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            print(f"\nFold {fold} | Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  "
                  f"time={epoch_time:.1f}s")
            val_metrics = evaluate(model, val_loader, f"fold{fold}-val")

            if val_metrics["f1"] > best_f1:
                best_f1 = val_metrics["f1"]
                torch.save(model.state_dict(),
                           os.path.join(fold_model_dir, "model.pt"))
                tokenizer.save_pretrained(fold_model_dir)
                print(f"  ✓ Meilleur modèle fold {fold} sauvegardé (F1={best_f1:.4f})")

            logf.write(json.dumps({
                "fold": fold, "epoch": epoch, "timestamp": timestamp,
                "duration_s": round(epoch_time, 2),
                "train_loss": round(avg_loss, 6),
                "val_f1":     round(val_metrics["f1"], 6),
            }, ensure_ascii=False) + "\n")
            logf.flush()

    # Évaluation finale sur le jeu de validation du fold (meilleur checkpoint)
    model.load_state_dict(torch.load(
        os.path.join(fold_model_dir, "model.pt"), map_location=device))
    final_metrics = evaluate(model, val_loader, f"fold{fold}-final", entries=val_ents)

    fold_results.append({
        "fold": fold,
        "train_size": len(train_idx),
        "val_size":   len(val_idx),
        "best_val_f1":     round(best_f1, 6),
        "final_f1":        round(final_metrics["f1"], 6),
        "final_precision": round(final_metrics["precision"], 6),
        "final_recall":    round(final_metrics["recall"], 6),
        "report_by_label": final_metrics["report"],
        "details":         final_metrics["details"],
    })
    with open(os.path.join(fold_log_dir, "results.json"), "w", encoding="utf-8") as f:
        json.dump(fold_results[-1], f, ensure_ascii=False, indent=2, cls=NumpyEncoder)




[3/5] Cross-validation 5-fold sur 845 entrées (test isolé)...

FOLD 1/5  (train=676, val=169)


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Paramètres entraînables : 14,184,203 / 278,052,107 (5.10%)

Fold 1 | Epoch 1/15  loss=2.6050  time=6.3s
  [fold1-val] F1=0.0000  P=0.0000  R=0.0000
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00       151
        date       0.00      0.00      0.00       164
        etat       0.00      0.00      0.00        33
    material       0.00      0.00      0.00       166
     ouvrage       0.00      0.00      0.00       210

   micro avg       0.00      0.00      0.00       724
   macro avg       0.00      0.00      0.00       724
weighted avg       0.00      0.00      0.00       724


Fold 1 | Epoch 2/15  loss=1.7994  time=5.3s
  [fold1-val] F1=0.1154  P=0.0983  R=0.1395
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00       151
        date       0.36      0.47      0.41       164
        etat       0.00      0.00      0.00        33
    material       0.03      0.05      0.04       166
 

...

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Paramètres entraînables : 14,184,203 / 278,052,107 (5.10%)

Fold 4 | Epoch 1/15  loss=2.5518  time=6.1s
  [fold4-val] F1=0.0000  P=0.0000  R=0.0000
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00       151
        date       0.00      0.00      0.00       164
        etat       0.00      0.00      0.00        33
    material       0.00      0.00      0.00       165
     ouvrage       0.00      0.00      0.00       210

   micro avg       0.00      0.00      0.00       723
   macro avg       0.00      0.00      0.00       723
weighted avg       0.00      0.00      0.00       723


Fold 4 | Epoch 2/15  loss=1.7113  time=6.1s
  [fold4-val] F1=0.1186  P=0.0989  R=0.1480
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00       151
        date       0.21      0.30      0.24       164
        etat       0.00      0.00      0.00        33
    material       0.12      0.21      0.15       165
 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Paramètres entraînables : 14,184,203 / 278,052,107 (5.10%)

Fold 5 | Epoch 1/15  loss=2.2508  time=6.1s
  [fold5-val] F1=0.0018  P=0.0029  R=0.0013
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00       165
        date       0.00      0.00      0.00       167
        etat       0.00      0.00      0.00        41
    material       0.00      0.00      0.00       172
     ouvrage       0.00      0.01      0.00       199

   micro avg       0.00      0.00      0.00       744
   macro avg       0.00      0.00      0.00       744
weighted avg       0.00      0.00      0.00       744

  ✓ Meilleur modèle fold 5 sauvegardé (F1=0.0018)

Fold 5 | Epoch 2/15  loss=1.6312  time=6.1s
  [fold5-val] F1=0.1475  P=0.1207  R=0.1895
              precision    recall  f1-score   support

      auteur       0.01      0.01      0.01       165
        date       0.36      0.56      0.44       167
        etat       0.00      0.00      0.00        41
    m

## Cell 1e — Résumé Cross-validation

F1 moyen ± écart-type, sauvegarde `cv_summary.json`.

In [ ]:
# ── 8. Résumé cross-validation ────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RÉSUMÉ CROSS-VALIDATION")
print('='*60)

f1_scores  = [r["final_f1"]        for r in fold_results]
pre_scores = [r["final_precision"]  for r in fold_results]
rec_scores = [r["final_recall"]     for r in fold_results]

cv_summary = {
    "model":        "XLM-RoBERTa partial unfreeze (2 dernières couches) + classification head",
    "strategy":     "2 derniers encoder layers dégelés, le reste de l'encodeur reste gelé",
    "n_unfrozen":   N_UNFROZEN,
    "epochs":       EPOCHS,
    "lr":           LR,
    "batch_size":   BATCH_SIZE,
    "max_len":      MAX_LEN,
    "k_folds":      K_FOLDS,
    "seed":         SEED,
    "cv_entries":   len(cv_entries),
    "test_entries":  len(test_entries),
    "cv_f1_mean":   round(np.mean(f1_scores), 6),
    "cv_f1_std":    round(np.std(f1_scores), 6),
    "cv_pre_mean":  round(np.mean(pre_scores), 6),
    "cv_rec_mean":  round(np.mean(rec_scores), 6),
    "fold_val_f1s": [round(f, 4) for f in f1_scores],
    "total_time_min": round((time.time() - cv_start) / 60, 2),
    "fold_results": fold_results,
    "timestamp":    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

print(f"  F1 moyen  : {cv_summary['cv_f1_mean']:.4f} ± {cv_summary['cv_f1_std']:.4f}")
print(f"  F1 par fold : {cv_summary['fold_val_f1s']}")

summary_path = os.path.join(LOG_DIR, "cv_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(cv_summary, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)
print(f"\n[Done] Résumé CV → {summary_path}")



RÉSUMÉ CROSS-VALIDATION
  F1 moyen  : 0.5171 ± 0.0226
  F1 par fold : [np.float64(0.5105), np.float64(0.5238), np.float64(0.5535), np.float64(0.5145), np.float64(0.4834)]

[Done] Résumé CV → /content/run3_xlmroberta_partial_unfreeze/logs/cv_summary.json


## Cell 1f — Évaluation finale sur test_all.json

Sélection du meilleur fold, évaluation sur les 93 entrées isolées.

In [ ]:
# ── 9. Évaluation finale sur test_all.json (93 entrées isolées) ───────────────
print(f"\n{'='*60}")
print("ÉVALUATION FINALE — test_all.json (93 entrées)")
print('='*60)

# Sélection du meilleur fold (F1 de validation le plus élevé)
best_fold_idx  = int(np.argmax([r["best_val_f1"] for r in fold_results]))
best_fold_num  = fold_results[best_fold_idx]["fold"]
best_fold_f1   = fold_results[best_fold_idx]["best_val_f1"]
best_model_dir = os.path.join(MODEL_DIR, f"fold_{best_fold_num}")
print(f"  Meilleur fold sélectionné : Fold {best_fold_num} (val F1={best_fold_f1:.4f})")

# Chargement du meilleur modèle
best_model = PartialUnfreezeXLMRoBERTaNER(MODEL_NAME, NUM_LABELS, n_unfrozen=N_UNFROZEN).to(device)
best_model.load_state_dict(torch.load(
    os.path.join(best_model_dir, "model.pt"), map_location=device))

# Dataset et DataLoader test
test_dataset_final = NERDataset(test_samples, tokenizer, MAX_LEN)
test_loader_final  = DataLoader(test_dataset_final, batch_size=BATCH_SIZE)

# Évaluation
test_final_metrics = evaluate(best_model, test_loader_final, "TEST FINAL", entries=test_entries)

# Sauvegarde
test_result = {
    "best_fold":       best_fold_num,
    "best_val_f1":     round(best_fold_f1, 6),
    "test_f1":         round(test_final_metrics["f1"], 6),
    "test_precision":  round(test_final_metrics["precision"], 6),
    "test_recall":     round(test_final_metrics["recall"], 6),
    "report_by_label": test_final_metrics["report"],
    "details":         test_final_metrics["details"],
    "timestamp":       datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}
test_result_path = os.path.join(LOG_DIR, "test_final_results.json")
with open(test_result_path, "w", encoding="utf-8") as f:
    json.dump(test_result, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)

print(f"\n[Done] Test F1 final : {test_result['test_f1']:.4f}")
print(f"[Done] Résultats test → {test_result_path}")



ÉVALUATION FINALE — test_all.json (93 entrées)
  Meilleur fold sélectionné : Fold 3 (val F1=0.5535)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [TEST FINAL] F1=0.4984  P=0.4283  R=0.5960
              precision    recall  f1-score   support

      auteur       0.35      0.53      0.42        79
        date       0.80      0.90      0.85        92
        etat       0.10      0.18      0.13        22
    material       0.58      0.67      0.62        91
     ouvrage       0.26      0.41      0.32       112

   micro avg       0.43      0.60      0.50       396
   macro avg       0.42      0.54      0.47       396
weighted avg       0.46      0.60      0.52       396


[Done] Test F1 final : 0.4984
[Done] Résultats test → /content/run3_xlmroberta_partial_unfreeze/logs/test_final_results.json


In [ ]:
# ── 打包 run3_xlmroberta_partial_unfreeze ────────────────────────────────────
import shutil, os

src_dir  = "/content/run3_xlmroberta_partial_unfreeze"
out_path = "/content/run3_xlmroberta_partial_unfreeze"  # .zip 自动添加

shutil.make_archive(out_path, "zip", "/content", "run3_xlmroberta_partial_unfreeze")
zip_size = os.path.getsize(out_path + ".zip") / (1024**2)
print(f"✓ Archive créée : {out_path}.zip  ({zip_size:.1f} MB)")

# 下载到本地
from google.colab import files
files.download(out_path + ".zip")

## Cell 2 — BiLSTM-CRF from scratch

**Stratégie** : BiLSTM-CRF entraîné **depuis zéro** (initialisation aléatoire), sans pré-entraînement. Utilise des embeddings de mots appris pendant l'entraînement.

**Hyperparamètres** :
- `epochs=30` : le Run 2 a montré que nos paramètres (epochs=50, lr=0.001) généralisaient mieux que les paramètres de l'article (epochs=5). Avec 938 entrées au lieu de 70, la convergence est plus rapide → on réduit à **30 epochs**.
- `lr=0.001` : nos paramètres du Run 2 ont mieux généralisé que lr=0.01 (article) sur le test set (F1=0.60 vs 0.52).
- `batch=8` : inchangé (adapté au corpus).
- Architecture : embed_dim=100, hidden_dim=256 (identique au Run 2).

**Données** : 938 entrées, cross-validation 5-fold.

**Sorties** → `/content/run3_bilstm_crf/`

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — BiLSTM-CRF FROM SCRATCH
# Cross-validation 5-fold sur 100 entrées
#
# Stratégie (recommandation encadrant) :
#   Entraîner BiLSTM-CRF depuis zéro (from scratch), sans pré-entraînement.
#   Sert de baseline classique NER sans transfert.
#
# Hyperparamètres (justification basée sur l'analyse du Run 2) :
#   EPOCHS=30   : Run 2 a montré que nos params (epochs=50) généralisaient
#                  mieux que l'article (epochs=5). Avec 100 entrées (vs 70
#                  en Run 2), la convergence est plus rapide → 30 suffit.
#   LR=0.001    : Run 2 : nos params (0.001) → test F1=0.60 vs article
#                  (0.01) → test F1=0.52. lr=0.001 conservé.
#   BATCH=8     : inchangé (adapté au corpus de ≤80 exemples par fold)
#   EMBED_DIM=100, HIDDEN_DIM=256 : même architecture que Run 2
#   K_FOLDS=5   : même protocole que Cell 1
#
# Données : train.json + val.json + test.json (100 entrées au total)
# ═══════════════════════════════════════════════════════════════

# -*- coding: utf-8 -*-
"""
prima_run3_bilstm_crf.py
========================
BiLSTM-CRF entraîné depuis zéro sur 100 entrées annotées.
Cross-validation 5-fold.

■ Architecture
  Embedding(vocab_size, 100) → BiLSTM(256 hidden) → Linear → CRF
  Identique au Run 2, sans modification de l'architecture.

■ Différence clé avec le Run 2
  Run 2 : 70 exemples d'entraînement, split fixe 70/20/10
  Run 3 : 80 exemples d'entraînement par fold, cross-validation 5-fold

■ Outputs
  logs/fold_{k}/training_log.jsonl
  logs/fold_{k}/results.json
  logs/cv_summary.json
  model/fold_{k}/model.pt + word2id.json
"""

import json
import os
import time
from datetime import datetime
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)


# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR   = "/content"  # fichiers uploadés via Cell 0
LOG_DIR    = "/content/run3_bilstm_crf/logs"
MODEL_DIR  = "/content/run3_bilstm_crf/model"

# ── Hyperparamètres (justification dans l'en-tête de cellule) ─────────────────
EPOCHS     = 30      # Run 2 : epochs=50 (nos params) > epochs=5 (article). Avec 100 entries → 30.
BATCH_SIZE = 8       # Inchangé
LR         = 0.001   # Run 2 : lr=0.001 → test F1=0.60 vs lr=0.01 → test F1=0.52
EMBED_DIM  = 100     # Identique au Run 2
HIDDEN_DIM = 256     # Identique au Run 2
DROPOUT    = 0.3     # Identique au Run 2
K_FOLDS    = 5
SEED       = 42

LABELS = ["auteur", "ouvrage", "material", "date", "etat"]
LABEL2ID = {"O": 0}
for lab in LABELS:
    LABEL2ID[f"B-{lab}"] = len(LABEL2ID)
    LABEL2ID[f"I-{lab}"] = len(LABEL2ID)
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)  # = 11

os.makedirs(LOG_DIR,   exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Config] Device : {device}")
print(f"[Config] BiLSTM-CRF from scratch")
print(f"[Config] Epochs={EPOCHS}, LR={LR}, Batch={BATCH_SIZE}, "
      f"Embed={EMBED_DIM}, Hidden={HIDDEN_DIM}, K={K_FOLDS}")


# ── 1. Conversion JSON → BIO ──────────────────────────────────────────────────
def json_to_bio(entries):
    """Identique au Run 2 (segmentation par espaces)."""
    samples = []
    for entry in entries:
        raw   = entry["raw"]
        spans = entry.get("spans", [])
        char_labels = ["O"] * len(raw)
        for span in spans:
            text  = span["text"]
            label = span["label"]
            idx   = raw.find(text)
            if idx == -1: continue
            char_labels[idx] = f"B-{label}"
            for i in range(idx + 1, idx + len(text)):
                char_labels[i] = f"I-{label}"
        tokens, labels = [], []
        pos = 0
        for word in raw.split():
            idx = raw.find(word, pos)
            if idx == -1:
                tokens.append(word); labels.append("O")
                pos += len(word); continue
            tokens.append(word)
            labels.append(char_labels[idx])
            pos = idx + len(word)
        samples.append((tokens, labels))
    return samples


# ── 2. Vocabulaire ────────────────────────────────────────────────────────────
def build_vocab(samples, min_freq=1):
    """
    Construit le vocabulaire à partir des données d'entraînement du fold.
    <PAD>=0 (padding), <UNK>=1 (mots inconnus au test).
    Important : le vocabulaire est reconstruit à chaque fold pour éviter
    toute fuite d'information du jeu de test vers l'entraînement.
    """
    freq = defaultdict(int)
    for tokens, _ in samples:
        for tok in tokens:
            freq[tok] += 1
    word2id = {"<PAD>": 0, "<UNK>": 1}
    for word, cnt in sorted(freq.items()):
        if cnt >= min_freq:
            word2id[word] = len(word2id)
    return word2id


# ── 3. Dataset ────────────────────────────────────────────────────────────────
class NERDataset(Dataset):
    """Dataset pour BiLSTM-CRF : entrées de longueur variable (padding dans collate_fn)."""
    def __init__(self, samples, word2id):
        self.data    = samples
        self.word2id = word2id

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]
        token_ids = [self.word2id.get(t, 1) for t in tokens]  # 1 = <UNK>
        label_ids = [LABEL2ID[l] for l in labels]
        return torch.tensor(token_ids, dtype=torch.long), \
               torch.tensor(label_ids, dtype=torch.long)


def collate_fn(batch):
    """Padding dynamique : pad à la longueur maximale du batch."""
    token_seqs, label_seqs = zip(*batch)
    max_len  = max(len(s) for s in token_seqs)
    token_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    label_ids = torch.zeros(len(batch), max_len, dtype=torch.long)
    mask      = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, (toks, labs) in enumerate(zip(token_seqs, label_seqs)):
        n = len(toks)
        token_ids[i, :n] = toks
        label_ids[i, :n] = labs
        mask[i, :n]      = True
    return token_ids, label_ids, mask


# ── 4. Modèle BiLSTM-CRF ─────────────────────────────────────────────────────
class CRF(nn.Module):
    """
    Couche CRF (Conditional Random Field).
    Modélise les dépendances entre étiquettes adjacentes (ex : I-X ne peut
    suivre qu'une étiquette B-X ou I-X). Améliore la cohérence des
    séquences prédites par rapport à un simple softmax.
    """
    def __init__(self, num_labels):
        super().__init__()
        self.num_labels   = num_labels
        # Matrice de transition : transitions[i, j] = score de passer de l'étiquette j à i
        self.transitions  = nn.Parameter(torch.randn(num_labels, num_labels))
        # Contraintes : pas de transition depuis/vers les tokens spéciaux
        self.start_transitions = nn.Parameter(torch.randn(num_labels))
        self.end_transitions   = nn.Parameter(torch.randn(num_labels))

    def _forward_alg(self, emissions, mask):
        """Algorithme forward pour calculer la partition (log-sum-exp)."""
        batch_size, seq_len, num_labels = emissions.size()
        # Score de départ
        score = self.start_transitions + emissions[:, 0]
        for i in range(1, seq_len):
            broadcast = score.unsqueeze(2)
            emit      = emissions[:, i].unsqueeze(1)
            next_score = broadcast + self.transitions + emit
            next_score = torch.logsumexp(next_score, dim=1)
            score      = torch.where(mask[:, i].unsqueeze(1), next_score, score)
        score = score + self.end_transitions
        return torch.logsumexp(score, dim=1)

    def _score_sentence(self, emissions, labels, mask):
        """Score de la séquence gold (numérateur de la CRF loss)."""
        batch_size, seq_len, _ = emissions.size()
        score = self.start_transitions[labels[:, 0]]
        score = score + emissions[:, 0].gather(1, labels[:, 0].unsqueeze(1)).squeeze(1)
        for i in range(1, seq_len):
            m = mask[:, i]
            t = self.transitions[labels[:, i], labels[:, i-1]]
            e = emissions[:, i].gather(1, labels[:, i].unsqueeze(1)).squeeze(1)
            score = score + (t + e) * m
        last_label_idx = mask.sum(dim=1) - 1
        last_labels    = labels.gather(1, last_label_idx.unsqueeze(1).clamp(min=0)).squeeze(1)
        score = score + self.end_transitions[last_labels]
        return score

    def neg_log_likelihood(self, emissions, labels, mask):
        """Loss CRF = log(partition) - score(gold)."""
        forward_score = self._forward_alg(emissions, mask)
        gold_score    = self._score_sentence(emissions, labels, mask)
        return (forward_score - gold_score).mean()

    def viterbi_decode(self, emissions, mask):
        """Décodage Viterbi : séquence d'étiquettes optimale."""
        batch_size, seq_len, num_labels = emissions.size()
        viterbi   = self.start_transitions + emissions[:, 0]
        backpointers = []
        for i in range(1, seq_len):
            broadcast    = viterbi.unsqueeze(2)
            next_scores  = broadcast + self.transitions
            best_scores, best_labels = next_scores.max(dim=1)
            next_viterbi = best_scores + emissions[:, i]
            viterbi      = torch.where(mask[:, i].unsqueeze(1), next_viterbi, viterbi)
            backpointers.append(best_labels)
        viterbi = viterbi + self.end_transitions
        _, best_last_label = viterbi.max(dim=1)
        best_path = [best_last_label]
        for bp in reversed(backpointers):
            best_last_label = bp.gather(1, best_last_label.unsqueeze(1)).squeeze(1)
            best_path.append(best_last_label)
        best_path.reverse()
        return torch.stack(best_path, dim=1)


class BiLSTM_CRF(nn.Module):
    """
    BiLSTM-CRF classique pour la NER.
    Architecture : Embedding → BiLSTM → Dropout → Linear → CRF

    Tous les paramètres sont initialisés aléatoirement (from scratch).
    Le modèle apprend les représentations de mots et les patterns
    séquentiels directement depuis les données PRIMA.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(
            embed_dim, hidden_dim // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim, num_labels)
        self.crf       = CRF(num_labels)

    def get_emissions(self, token_ids, mask):
        emb = self.dropout(self.embedding(token_ids))
        lengths = mask.sum(dim=1).cpu()
        packed  = nn.utils.rnn.pack_padded_sequence(
            emb, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        # Padding à la longueur originale si nécessaire
        if out.size(1) < token_ids.size(1):
            pad = torch.zeros(
                out.size(0), token_ids.size(1) - out.size(1), out.size(2),
                device=out.device)
            out = torch.cat([out, pad], dim=1)
        return self.fc(self.dropout(out))

    def neg_log_likelihood(self, token_ids, label_ids, mask):
        emissions = self.get_emissions(token_ids, mask)
        return self.crf.neg_log_likelihood(emissions, label_ids, mask)

    def viterbi_decode(self, token_ids, mask):
        emissions = self.get_emissions(token_ids, mask)
        return self.crf.viterbi_decode(emissions, mask)


# ── 5. Évaluation ─────────────────────────────────────────────────────────────
def bio_to_spans(tokens, bio):
    spans = []
    cur_text, cur_label = [], None
    for tok, tag in zip(tokens, bio):
        if tag.startswith("B-"):
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text  = [tok]; cur_label = tag[2:]
        elif tag.startswith("I-") and cur_text:
            cur_text.append(tok)
        else:
            if cur_text:
                spans.append({"text": " ".join(cur_text), "label": cur_label})
            cur_text, cur_label = [], None
    if cur_text:
        spans.append({"text": " ".join(cur_text), "label": cur_label})
    return spans


def evaluate(model, loader, split_name="test", entries=None):
    model.eval()
    all_preds, all_trues = [], []
    with torch.no_grad():
        for token_ids, label_ids, mask in loader:
            token_ids = token_ids.to(device)
            mask      = mask.to(device)
            preds     = model.viterbi_decode(token_ids, mask)
            for pred_seq, true_seq, m in zip(
                    preds.cpu().tolist(), label_ids.tolist(), mask.cpu().tolist()):
                p_tags = [ID2LABEL[p] for p, valid in zip(pred_seq, m) if valid]
                t_tags = [ID2LABEL[t] for t, valid in zip(true_seq, m) if valid]
                all_preds.append(p_tags)
                all_trues.append(t_tags)
    f1  = f1_score(all_trues, all_preds, zero_division=0)
    pre = precision_score(all_trues, all_preds, zero_division=0)
    rec = recall_score(all_trues, all_preds, zero_division=0)
    report = classification_report(all_trues, all_preds, output_dict=True, zero_division=0)
    print(f"  [{split_name}] F1={f1:.4f}  P={pre:.4f}  R={rec:.4f}")
    print(classification_report(all_trues, all_preds, zero_division=0))
    details = []
    if entries is not None:
        for entry, gold_bio, pred_bio in zip(entries, all_trues, all_preds):
            tokens = entry["raw"].split()
            details.append({
                "entry_id":   entry["entry_id"],
                "raw":        entry["raw"],
                "gold_spans": bio_to_spans(tokens, gold_bio),
                "pred_spans": bio_to_spans(tokens, pred_bio),
            })
    return {"f1": f1, "precision": pre, "recall": rec, "report": report, "details": details}


# ── 6. Chargement des données (100 entrées) ───────────────────────────────────
print("\n[1/4] Chargement des 100 entrées annotées...")
all_entries = []
for split in ["train", "val", "test"]:
    path = os.path.join(DATA_DIR, f"{split}_all.json")
    with open(path, encoding="utf-8") as f:
        part = json.load(f)
    all_entries.extend(part)
    print(f"  {split}: {len(part)} entrées")
print(f"  Total : {len(all_entries)} entrées")

all_samples = json_to_bio(all_entries)


# ── 7. Cross-validation 5-fold ────────────────────────────────────────────────
print(f"\n[2/4] Cross-validation {K_FOLDS}-fold (BiLSTM-CRF from scratch)...")
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

fold_results = []
cv_start = time.time()

for fold, (train_idx, test_idx) in enumerate(kf.split(range(len(all_entries))), start=1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{K_FOLDS}  (train={len(train_idx)}, test={len(test_idx)})")
    print('='*60)

    fold_log_dir   = os.path.join(LOG_DIR,   f"fold_{fold}")
    fold_model_dir = os.path.join(MODEL_DIR, f"fold_{fold}")
    os.makedirs(fold_log_dir,   exist_ok=True)
    os.makedirs(fold_model_dir, exist_ok=True)

    train_samp = [all_samples[i] for i in train_idx]
    test_samp  = [all_samples[i] for i in test_idx]
    test_ents  = [all_entries[i] for i in test_idx]

    # Vocabulaire construit sur les données d'entraînement du fold uniquement
    # (évite la fuite d'information depuis le test set)
    word2id = build_vocab(train_samp)
    print(f"  Vocab size : {len(word2id)} mots")

    train_dataset = NERDataset(train_samp, word2id)
    test_dataset  = NERDataset(test_samp,  word2id)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                               shuffle=True, collate_fn=collate_fn)
    test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, collate_fn=collate_fn)

    # Initialisation depuis zéro (from scratch)
    model = BiLSTM_CRF(
        vocab_size  = len(word2id),
        embed_dim   = EMBED_DIM,
        hidden_dim  = HIDDEN_DIM,
        num_labels  = NUM_LABELS,
        dropout     = DROPOUT,
    ).to(device)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Paramètres totaux : {trainable:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    log_path  = os.path.join(fold_log_dir, "training_log.jsonl")
    best_f1   = 0.0

    with open(log_path, "w", encoding="utf-8") as logf:
        for epoch in range(1, EPOCHS + 1):
            epoch_start = time.time()
            model.train()
            total_loss = 0.0

            for token_ids, label_ids, mask in train_loader:
                token_ids = token_ids.to(device)
                label_ids = label_ids.to(device)
                mask      = mask.to(device)

                loss = model.neg_log_likelihood(token_ids, label_ids, mask)
                total_loss += loss.item()

                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping : CRF peut avoir des gradients instables
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

            avg_loss   = total_loss / len(train_loader)
            epoch_time = time.time() - epoch_start
            timestamp  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            print(f"\nFold {fold} | Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  "
                  f"time={epoch_time:.1f}s")
            test_metrics = evaluate(model, test_loader, f"fold{fold}-test")

            if test_metrics["f1"] > best_f1:
                best_f1 = test_metrics["f1"]
                torch.save(model.state_dict(),
                           os.path.join(fold_model_dir, "model.pt"))
                with open(os.path.join(fold_model_dir, "word2id.json"), "w") as f:
                    json.dump(word2id, f, ensure_ascii=False)
                print(f"  ✓ Meilleur modèle fold {fold} sauvegardé (F1={best_f1:.4f})")

            logf.write(json.dumps({
                "fold": fold, "epoch": epoch, "timestamp": timestamp,
                "duration_s": round(epoch_time, 2),
                "train_loss": round(avg_loss, 6),
                "test_f1":    round(test_metrics["f1"], 6),
            }, ensure_ascii=False) + "\n")
            logf.flush()

    # Évaluation finale
    model.load_state_dict(torch.load(
        os.path.join(fold_model_dir, "model.pt"), map_location=device))
    final_metrics = evaluate(model, test_loader, f"fold{fold}-final", entries=test_ents)

    fold_results.append({
        "fold": fold,
        "train_size": len(train_idx),
        "test_size":  len(test_idx),
        "vocab_size": len(word2id),
        "best_f1":         round(best_f1, 6),
        "final_f1":        round(final_metrics["f1"], 6),
        "final_precision": round(final_metrics["precision"], 6),
        "final_recall":    round(final_metrics["recall"], 6),
        "report_by_label": final_metrics["report"],
        "details":         final_metrics["details"],
    })
    with open(os.path.join(fold_log_dir, "results.json"), "w", encoding="utf-8") as f:
        json.dump(fold_results[-1], f, ensure_ascii=False, indent=2, cls=NumpyEncoder)


# ── 8. Résumé cross-validation ────────────────────────────────────────────────
print(f"\n{'='*60}")
print("RÉSUMÉ CROSS-VALIDATION")
print('='*60)

f1_scores  = [r["final_f1"]        for r in fold_results]
pre_scores = [r["final_precision"]  for r in fold_results]
rec_scores = [r["final_recall"]     for r in fold_results]

cv_summary = {
    "model":        "BiLSTM-CRF from scratch",
    "strategy":     "Entraîné depuis zéro, sans pré-entraînement",
    "epochs":       EPOCHS,
    "lr":           LR,
    "batch_size":   BATCH_SIZE,
    "embed_dim":    EMBED_DIM,
    "hidden_dim":   HIDDEN_DIM,
    "k_folds":      K_FOLDS,
    "seed":         SEED,
    "total_entries": len(all_entries),
    "cv_f1_mean":   round(np.mean(f1_scores), 6),
    "cv_f1_std":    round(np.std(f1_scores), 6),
    "cv_pre_mean":  round(np.mean(pre_scores), 6),
    "cv_rec_mean":  round(np.mean(rec_scores), 6),
    "fold_f1s":     [round(f, 4) for f in f1_scores],
    "total_time_min": round((time.time() - cv_start) / 60, 2),
    "fold_results": fold_results,
    "timestamp":    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

print(f"  F1 moyen  : {cv_summary['cv_f1_mean']:.4f} ± {cv_summary['cv_f1_std']:.4f}")
print(f"  F1 par fold : {cv_summary['fold_f1s']}")

summary_path = os.path.join(LOG_DIR, "cv_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(cv_summary, f, ensure_ascii=False, indent=2, cls=NumpyEncoder)
print(f"\n[Done] Résumé CV → {summary_path}")

[Config] Device : cuda
[Config] BiLSTM-CRF from scratch
[Config] Epochs=30, LR=0.001, Batch=8, Embed=100, Hidden=256, K=5

[1/4] Chargement des 100 entrées annotées...
  train: 657 entrées
  val: 188 entrées
  test: 93 entrées
  Total : 938 entrées

[2/4] Cross-validation 5-fold (BiLSTM-CRF from scratch)...

FOLD 1/5  (train=750, test=188)
  Vocab size : 3697 mots
  Paramètres totaux : 608,190

Fold 1 | Epoch 1/30  loss=26.4670  time=4.0s
  [fold1-test] F1=0.3965  P=0.4227  R=0.3734
              precision    recall  f1-score   support

      auteur       0.00      0.00      0.00       158
        date       0.94      0.92      0.93       188
        etat       0.18      0.34      0.23        38
    material       0.34      0.37      0.36       188
     ouvrage       0.19      0.19      0.19       226

   micro avg       0.42      0.37      0.40       798
   macro avg       0.33      0.36      0.34       798
weighted avg       0.37      0.37      0.37       798

  ✓ Meilleur modèle fol